In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("❌ GPU is NOT available")

CUDA available: True
GPU: Tesla T4
GPU count: 2


In [2]:
from pathlib import Path

input_root = Path("/kaggle/input")
gguf_files = sorted(input_root.rglob("*.gguf"))

print("Input root exists:", input_root.exists())
print("GGUF shards found:", len(gguf_files))

for path in gguf_files:
    print(path)
    print(f"  size: {path.stat().st_size:,} bytes")
    print(f"  size: {path.stat().st_size / 1024**3:.3f} GiB")

expected_names = {
    "qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf",
    "qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf",
}
found_names = {path.name for path in gguf_files}
missing = expected_names - found_names

if missing:
    raise FileNotFoundError(f"Missing expected model shards: {sorted(missing)}")

model_path = next(path for path in gguf_files if "00001-of-00002" in path.name)
total_size = sum(path.stat().st_size for path in gguf_files)

print(f"Model path: {model_path}")
print(f"Total GGUF size: {total_size / 1024**3:.3f} GiB")
print("Model file access: OK")

Input root exists: True
GGUF shards found: 2
/kaggle/input/datasets/keshav0210/qwen25-7b-instruct-q4-km-local/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
  size: 3,993,201,344 bytes
  size: 3.719 GiB
/kaggle/input/datasets/keshav0210/qwen25-7b-instruct-q4-km-local/qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf
  size: 689,872,288 bytes
  size: 0.642 GiB
Model path: /kaggle/input/datasets/keshav0210/qwen25-7b-instruct-q4-km-local/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
Total GGUF size: 4.361 GiB
Model file access: OK


In [3]:
import os
from pathlib import Path

print("Remote Python:", os.sys.executable)
print("Remote working directory:", Path.cwd())
print("Project source present:", (Path.cwd() / "src" / "mastercard_defence").exists())
print("Working directory entries:")
for path in sorted(Path.cwd().iterdir()):
    print(" ", path)

Remote Python: /usr/bin/python3
Remote working directory: /kaggle/working
Project source present: False
Working directory entries:
  /kaggle/working/.virtual_documents
  /kaggle/working/mastercard_hackathon


In [7]:
import subprocess
import sys
from pathlib import Path

project_dir = Path("/kaggle/working/mastercard_hackathon")
repository_url = "https://github.com/keshav-0210/mastercard_hackathon.git"

if not project_dir.exists():
    subprocess.run(
        ["git", "clone", repository_url, str(project_dir)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only"], check=True)

source_root = project_dir / "src"
sys.path.insert(0, str(source_root))

required_paths = [
    source_root / "mastercard_defence",
    project_dir / "config" / "default.yaml",
    project_dir / "data" / "knowledge_base",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing synced project paths: {missing}")

print("Project synced:", project_dir)
print("Source import path:", source_root)
print("Project sync: OK")

Updating bc30b70..c9849c3
Fast-forward
 src/mastercard_defence/agents.py | 3 +++
 1 file changed, 3 insertions(+)
Project synced: /kaggle/working/mastercard_hackathon
Source import path: /kaggle/working/mastercard_hackathon/src
Project sync: OK


From https://github.com/keshav-0210/mastercard_hackathon
   bc30b70..c9849c3  main       -> origin/main


In [20]:
from mastercard_defence.rag import LocalKnowledgeBase

knowledge_base = LocalKnowledgeBase(str(project_dir / "data" / "knowledge_base"))
evidence = knowledge_base.retrieve(
    "generative AI risk threats mitigation payment privacy synthetic data fidelity",
    top_k=4,
)

print("Knowledge documents:", len(knowledge_base.documents))
print("Retrieved evidence:")
for item in evidence:
    print(f"- {item.source_id}: {item.title}")

assert len(knowledge_base.documents) == 7
assert len(evidence) == 4
assert all(item.title for item in evidence)
print("Compliance-safe RAG corpus: OK")

Knowledge documents: 7
Retrieved evidence:
- nist_privacy_framework: NIST Privacy Framework
- pci_payment_security_overview: PCI Security Standards Council payment-security overview
- nist_genai_profile_2024: Artificial Intelligence Risk Management Framework: Generative Artificial Intelligence Profile
- synthetic_data_evaluation_principles: Synthetic-data evaluation principles for this project
Compliance-safe RAG corpus: OK


In [4]:
import os

os.environ["RUN_MODE"] = "KAGGLE_GPU"
os.environ["MODEL_PATH"] = str(model_path)

from mastercard_defence.loop import load_config
from mastercard_defence.runtime import require_model

config_path = project_dir / "config" / "default.yaml"
config = load_config(str(config_path))
validated_model_path = require_model(config)

try:
    import llama_cpp
    print("llama_cpp version:", getattr(llama_cpp, "__version__", "unknown"))
    print("Project imports: OK")
    print("Model configuration: OK")
except ImportError:
    print("llama_cpp is not installed in the Kaggle kernel")

llama_cpp version: 0.3.35
Project imports: OK
Model configuration: OK


In [19]:
%env CMAKE_ARGS=-DGGML_CUDA=on
%pip uninstall -y llama-cpp-python
%pip install --no-cache-dir --no-binary=llama-cpp-python llama-cpp-python

env: CMAKE_ARGS=-DGGML_CUDA=on
Found existing installation: llama_cpp_python 0.3.35
Uninstalling llama_cpp_python-0.3.35:
  Successfully uninstalled llama_cpp_python-0.3.35
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 282.7 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
Failed to build llama-cpp-python
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (llama-cpp-python)
Note: you may need to restart the kernel t

In [12]:
import subprocess
import torch

print("Torch CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("NVIDIA runtime:")
subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], check=False)

Torch CUDA version: 12.8
CUDA available: True
GPU: Tesla T4
NVIDIA runtime:
Tesla T4, 580.159.04
Tesla T4, 580.159.04


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], returncode=0)

In [20]:
%pip install --no-cache-dir --prefer-binary --index-url https://abetlen.github.io/llama-cpp-python/whl/cu128 --extra-index-url https://pypi.org/simple llama-cpp-python

Looking in indexes: https://abetlen.github.io/llama-cpp-python/whl/cu128, https://pypi.org/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 239.9 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
Failed to build llama-cpp-python
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (llama-cpp-python)
Note: you may need to restart the kernel to use updated packages.


In [21]:
%pip install --no-cache-dir --prefer-binary --index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 --extra-index-url https://pypi.org/simple llama-cpp-python

Looking in indexes: https://abetlen.github.io/llama-cpp-python/whl/cu124, https://pypi.org/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 267.9 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [5]:
import json
import time

from llama_cpp import llama_supports_gpu_offload
from mastercard_defence.llm import SharedLocalLLM

print("llama.cpp GPU offload support:", llama_supports_gpu_offload())
if not llama_supports_gpu_offload():
    raise RuntimeError("Installed llama.cpp does not support GPU offload")

llm = SharedLocalLLM(config)
start = time.perf_counter()
response = llm.complete_json(
    system_prompt="You are a payment-security research assistant. Return only valid JSON.",
    user_prompt=(
        "Create one synthetic payment-security attack hypothesis. "
        "Return exactly the keys attack_family, scenario, and research_direction."
    ),
)
elapsed = time.perf_counter() - start

print("Model JSON response:")
print(json.dumps(response, indent=2))
print(f"Inference time: {elapsed:.2f} seconds")
print("Qwen GPU inference smoke test: OK")

ggml_cuda_init: found 2 CUDA devices (Total VRAM: 29823 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14911 MiB
  Device 1: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14911 MiB


llama.cpp GPU offload support: True
Model JSON response:
{
  "attack_family": "Man-in-the-Middle",
  "scenario": "An attacker intercepts the communication between a mobile device and the payment gateway to modify the transaction details, such as the amount or the recipient's account information, before the data reaches the intended recipient.",
  "research_direction": "Developing encryption methods and secure communication protocols to prevent data interception and ensure the integrity of payment transactions."
}
Inference time: 45.93 seconds
Qwen GPU inference smoke test: OK


In [16]:
import json
import time

start = time.perf_counter()
response = llm._load().create_chat_completion(
    messages=[
        {
            "role": "system",
            "content": "You are a payment-security research assistant. Return only valid JSON.",
        },
        {
            "role": "user",
            "content": (
                "Create one synthetic payment-security attack hypothesis. "
                "Return exactly the keys attack_family, scenario, and research_direction."
            ),
        },
    ],
    response_format={"type": "json_object"},
    temperature=0.2,
    max_tokens=300,
)
content = response["choices"][0]["message"]["content"]
parsed_response = json.loads(content)
print(json.dumps(parsed_response, indent=2))
print(f"Inference time: {time.perf_counter() - start:.2f} seconds")
print("Qwen GPU inference smoke test: OK")

{
  "attack_family": "Man-in-the-Middle",
  "scenario": "An attacker intercepts the communication between a mobile app and the payment gateway by exploiting a weak SSL/TLS implementation, allowing them to read and modify payment data.",
  "research_direction": "Developing more robust SSL/TLS configurations and implementing additional security measures such as certificate pinning to prevent such attacks."
}
Inference time: 6.63 seconds
Qwen GPU inference smoke test: OK


In [23]:
import os
import subprocess
import sys

subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only"], check=True)
os.chdir(project_dir)

for module_name in list(sys.modules):
    if module_name == "mastercard_defence" or module_name.startswith("mastercard_defence."):
        del sys.modules[module_name]

sys.path.insert(0, str(project_dir / "src"))
from mastercard_defence.loop import ClosedLoop, load_config

os.environ["RUN_MODE"] = "KAGGLE_GPU"
os.environ["MODEL_PATH"] = str(model_path)
config = load_config("config/default.yaml")
loop = ClosedLoop(config)
try:
    results = loop.run(rounds=3)
finally:
    loop.close()

assert len(results) == 3
assert len({result["hypothesis"].attack_family for result in results}) == 3
assert all(result["hypothesis"].evidence for result in results)
assert all(result["detection"]["evaluation_protocol"] == "unseen_attack_rows_and_legitimate_holdout" for result in results)
assert all("behavioural_signal_delta" in result["fidelity"] for result in results)
assert all("unique_row_ratio" in result["diversity"] for result in results)
assert all("novelty_score" in result["novelty"] for result in results)
assert [result["weakness"].round_id for result in results] == [1, 2, 3]

print("Rounds completed:", len(results))
print("Agent backend:", type(loop.agents).__name__)
print("Attack families:", [result["hypothesis"].attack_family for result in results])
print("Detection F1:", [round(result["detection"]["f1"], 3) for result in results])
print("Fidelity plausibility:", [result["fidelity"]["behavioural_plausibility"] for result in results])
print("Novelty scores:", [result["novelty"]["novelty_score"] for result in results])
print("Diversity:", [result["diversity"] for result in results])
print("Evaluation protocol:", results[0]["detection"]["evaluation_protocol"])
print("Agent 3 -> Memory -> Agent 1 loop: OK")

Already up to date.
Rounds completed: 3
Agent backend: QwenAgents
Attack families: ['social_engineering', 'trusted_device', 'account_takeover']
Detection F1: [0.826, 0.734, 0.769]
Fidelity plausibility: [np.float64(0.4706), np.float64(0.5399), np.float64(0.4793)]
Novelty scores: [0.9446, 0.9517, 0.9561]
Diversity: [{'attack_family_count': 1, 'channel_count': 3, 'unique_row_ratio': 1.0, 'numeric_feature_mean_count': 24}, {'attack_family_count': 1, 'channel_count': 3, 'unique_row_ratio': 1.0, 'numeric_feature_mean_count': 24}, {'attack_family_count': 1, 'channel_count': 3, 'unique_row_ratio': 1.0, 'numeric_feature_mean_count': 24}]
Evaluation protocol: unseen_attack_rows_and_legitimate_holdout
Agent 3 -> Memory -> Agent 1 loop: OK


In [ ]:
from mastercard_defence.agents import QwenAgents
from mastercard_defence.llm import SharedLocalLLM
from mastercard_defence.loop import ClosedLoop, load_config

os.environ["RUN_MODE"] = "KAGGLE_GPU"
os.environ["MODEL_PATH"] = str(model_path)
os.chdir(project_dir)
config = load_config("config/default.yaml")
shared_llm = SharedLocalLLM(config)
qwen_agents = QwenAgents(config, llm=shared_llm)
loop = ClosedLoop(config, agents=qwen_agents)
try:
    results = loop.run(rounds=3)
finally:
    loop.close()

assert len(results) == 3
assert all(result["hypothesis"].evidence for result in results)
assert all(result["detection"]["evaluation_protocol"] == "unseen_attack_rows_and_legitimate_holdout" for result in results)
assert all("behavioural_signal_delta" in result["fidelity"] for result in results)
assert all("unique_row_ratio" in result["diversity"] for result in results)
assert results[0]["weakness"].round_id == 1
assert results[1]["weakness"].round_id == 2
assert results[2]["weakness"].round_id == 3

print("Rounds completed:", len(results))
print("Agent backend:", type(loop.agents).__name__)
print("Attack families:", [result["hypothesis"].attack_family for result in results])
print("Detection F1:", [round(result["detection"]["f1"], 3) for result in results])
print("Fidelity plausibility:", [result["fidelity"]["behavioural_plausibility"] for result in results])
print("Diversity:", [result["diversity"] for result in results])
print("Agent 3 -> Memory -> Agent 1 loop: OK")

ValidationError: 1 validation error for AttackSpecification
realism_constraints
  Input should be a valid list [type=list_type, input_value={'transaction_volume': 'F...e transaction patterns'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type

In [18]:
import subprocess

print("Git revision:")
subprocess.run(["git", "-C", str(project_dir), "log", "-1", "--oneline"], check=True)
agent_source = (project_dir / "src" / "mastercard_defence" / "agents.py").read_text(encoding="utf-8")
print("QwenAgents present in Kaggle source:", "class QwenAgents" in agent_source)
print("QwenAgents selected in current process:", type(loop.agents).__name__)

Git revision:
4f25c38 Add first-cut AI Defence Lab pipeline
QwenAgents present in Kaggle source: False
QwenAgents selected in current process: HeuristicAgents


In [5]:
%pip install --no-cache-dir ctgan

import numpy as np
import pandas as pd
import torch
from ctgan import CTGAN

rng = np.random.default_rng(20260821)
rows = []
for family in ["legitimate", "account_takeover", "trusted_device"]:
    count = 30
    rows.extend({
        "amount": float(rng.lognormal(3.3 if family == "legitimate" else 4.0, 0.5)),
        "hour": int(rng.integers(0, 24)),
        "device_change": int(rng.binomial(1, 0.08 if family == "legitimate" else 0.25)),
        "channel": str(rng.choice(["web", "mobile", "card_present"])),
        "attack_family": family,
    } for _ in range(count))
training = pd.DataFrame(rows)

ctgan = CTGAN(embedding_dim=32, generator_dim=(64, 64), discriminator_dim=(64, 64), batch_size=30, epochs=1, pac=10, verbose=False, cuda=torch.cuda.is_available())
ctgan.set_random_state(20260821)
ctgan.fit(training, discrete_columns=["hour", "device_change", "channel", "attack_family"])
conditioned = []
for attempt in range(8):
    candidate = ctgan.sample(30, condition_column="attack_family", condition_value="trusted_device")
    matched = candidate[candidate["attack_family"] == "trusted_device"]
    conditioned.append(matched)
    if sum(len(batch) for batch in conditioned) >= 10:
        break
samples = pd.concat(conditioned, ignore_index=True).iloc[:10]
assert len(samples) == 10
assert set(samples["attack_family"]) == {"trusted_device"}
print("CUDA:", torch.cuda.is_available())
print("CTGAN device smoke: OK")
print("Strict conditioned rows:", len(samples))
print("Strict conditioned families:", sorted(samples["attack_family"].unique().tolist()))

Note: you may need to restart the kernel to use updated packages.


/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


CUDA: True
CTGAN device smoke: OK
Strict conditioned rows: 10
Strict conditioned families: ['trusted_device']


In [12]:
import os
import subprocess
import sys
import tempfile

subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only"], check=True)
os.chdir(project_dir)
for module_name in list(sys.modules):
    if module_name == "mastercard_defence" or module_name.startswith("mastercard_defence."):
        del sys.modules[module_name]
sys.path.insert(0, str(project_dir / "src"))
from mastercard_defence.loop import ClosedLoop, load_config

os.environ["RUN_MODE"] = "LOCAL"
config = load_config("config/default.yaml")
config["paths"]["memory_db"] = tempfile.mktemp(suffix=".sqlite")
config["generator_backend"] = "ctgan"
config["generator_epochs"] = 1
config["generator_training_attack_size"] = 10
config["generator_training_reference_size"] = 20
config["pipeline"]["synthetic_transactions"] = 100
config["pipeline"]["max_generated_attacks"] = 10

loop = ClosedLoop(config)
try:
    suite = loop.run_robustness_suite(seeds=2, rounds=2)
finally:
    loop.close()

assert suite["seed_count"] == 2
assert suite["rounds"] == 2
assert [run["seed"] for run in suite["by_seed"]] == [20260821, 20260822]
print("Integrated CTGAN smoke: OK")
print("CUDA:", torch.cuda.is_available())
print("Agent backend: HeuristicAgents")
print("Seeds:", [run["seed"] for run in suite["by_seed"]])
print("Rounds per seed:", [len(run["results"]) for run in suite["by_seed"]])
print("F1:", [round(result["detection"]["f1"], 3) for run in suite["by_seed"] for result in run["results"]])

Already up to date.


/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


[robustness] starting suite: seeds=2, rounds=2
[robustness] starting seed 20260821 with family plan: ['account_takeover', 'social_engineering']
[seed=20260821] starting 2-round run with family plan: ['account_takeover', 'social_engineering']
[seed=20260821] round 1/2 starting
[seed=20260821] round 1/2 complete | family=account_takeover | f1=0.3750 | novelty=1.0000
[seed=20260821] round 2/2 starting
[seed=20260821] round 2/2 complete | family=social_engineering | f1=0.5556 | novelty=0.9556
[seed=20260821] run complete. Total rounds: 2
[robustness] completed seed 20260821 with 2 rounds
[robustness] starting seed 20260822 with family plan: ['account_takeover', 'cross_channel_anomaly']
[seed=20260822] starting 2-round run with family plan: ['account_takeover', 'cross_channel_anomaly']
[seed=20260822] round 1/2 starting
[seed=20260822] round 1/2 complete | family=account_takeover | f1=0.3077 | novelty=0.9302
[seed=20260822] round 2/2 starting
[seed=20260822] round 2/2 complete | family=cros

In [10]:
from pathlib import Path
import subprocess

print(subprocess.run(["git", "-C", str(project_dir), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip())
learned_path = project_dir / "src" / "mastercard_defence" / "learned_generator.py"
print("learned_generator path:", learned_path)
print("family column preserved:", "attack_rows.append(rows[MODEL_COLUMNS])" in learned_path.read_text(encoding="utf-8"))
print("training families:", sorted(__import__("mastercard_defence.learned_generator", fromlist=["build_training_corpus"]).build_training_corpus(20260821, attack_size=2, reference_size=2)["attack_family"].unique().tolist()))

b1a0fca3834391f9c2e47da2f6a60b9147207123
learned_generator path: /kaggle/working/mastercard_hackathon/src/mastercard_defence/learned_generator.py
family column preserved: False


TypeError: '<' not supported between instances of 'float' and 'str'

In [15]:
import json
import os
import sys
import tempfile
from datetime import datetime, timezone

os.chdir(project_dir)
for module_name in list(sys.modules):
    if module_name == "mastercard_defence" or module_name.startswith("mastercard_defence."):
        del sys.modules[module_name]
sys.path.insert(0, str(project_dir / "src"))
from mastercard_defence.loop import ClosedLoop, load_config

os.environ["RUN_MODE"] = "LOCAL"
config = load_config("config/default.yaml")
config["paths"]["memory_db"] = tempfile.mktemp(suffix=".sqlite")
config["generator_backend"] = "ctgan"
config["generator_epochs"] = 5
config["pipeline"]["synthetic_transactions"] = 400
config["pipeline"]["max_generated_attacks"] = 80

loop = ClosedLoop(config)
try:
    suite = loop.run_robustness_suite(seeds=3, rounds=5)
finally:
    loop.close()

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
result_path = project_dir / "artifacts" / f"ctgan_robustness_results_{run_stamp}.json"
result_path.parent.mkdir(parents=True, exist_ok=True)
result_path.write_text(json.dumps({
    "run_timestamp_utc": run_stamp,
    "generator_backend": "conditional_ctgan",
    "agent_backend": "HeuristicAgents",
    "seed_count": suite["seed_count"],
    "rounds": suite["rounds"],
    "summary": suite["summary"],
    "by_seed": [
        {"seed": run["seed"], "results": [
            {"round": item["round"], "attack_family": item["specification"].attack_family,
             "detection": item["detection"], "fidelity": item["fidelity"],
             "diversity": item["diversity"], "novelty": item["novelty"]}
            for item in run["results"]
        ]}
        for run in suite["by_seed"]
    ],
}, indent=2, default=str), encoding="utf-8")
print("FULL_CTGAN_OK", suite["seed_count"], suite["rounds"])
print("CUDA:", torch.cuda.is_available())
print("Results saved:", result_path)
print("Summary:", suite["summary"])

/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


[robustness] starting suite: seeds=3, rounds=5
[robustness] starting seed 20260821 with family plan: ['account_takeover', 'social_engineering', 'trusted_device', 'beneficiary_manipulation', 'cross_channel_anomaly']
[seed=20260821] starting 5-round run with family plan: ['account_takeover', 'social_engineering', 'trusted_device', 'beneficiary_manipulation', 'cross_channel_anomaly']
[seed=20260821] round 1/5 starting
[seed=20260821] round 1/5 complete | family=account_takeover | f1=0.7576 | novelty=1.0000
[seed=20260821] round 2/5 starting
[seed=20260821] round 2/5 complete | family=social_engineering | f1=0.8029 | novelty=0.9556
[seed=20260821] round 3/5 starting
[seed=20260821] round 3/5 complete | family=trusted_device | f1=0.8000 | novelty=0.9302
[seed=20260821] round 4/5 starting
[seed=20260821] round 4/5 complete | family=beneficiary_manipulation | f1=0.7647 | novelty=0.9535
[seed=20260821] round 5/5 starting
[seed=20260821] round 5/5 complete | family=cross_channel_anomaly | f1=0.

In [8]:
import os
import sys
import tempfile

os.chdir(project_dir)
for module_name in list(sys.modules):
    if module_name == "mastercard_defence" or module_name.startswith("mastercard_defence."):
        del sys.modules[module_name]
sys.path.insert(0, str(project_dir / "src"))
from mastercard_defence.agents import QwenAgents
from mastercard_defence.llm import SharedLocalLLM
from mastercard_defence.loop import ClosedLoop, load_config

os.environ["RUN_MODE"] = "KAGGLE_GPU"
os.environ["MODEL_PATH"] = str(model_path)
config = load_config("config/default.yaml")
config["paths"]["memory_db"] = tempfile.mktemp(suffix=".sqlite")
config["generator_backend"] = "ctgan"
config["generator_epochs"] = 1
config["generator_training_attack_size"] = 10
config["generator_training_reference_size"] = 20
config["pipeline"]["synthetic_transactions"] = 100
config["pipeline"]["max_generated_attacks"] = 10

qwen_agents = QwenAgents(config, llm=SharedLocalLLM(config))
loop = ClosedLoop(config, agents=qwen_agents)
try:
    suite = loop.run_robustness_suite(seeds=2, rounds=2)
finally:
    loop.close()

assert suite["seed_count"] == 2
assert suite["rounds"] == 2
assert [run["seed"] for run in suite["by_seed"]] == [20260821, 20260822]
assert all(result["hypothesis"].evidence for run in suite["by_seed"] for result in run["results"])
print("Integrated Qwen + CTGAN smoke: OK")
print("Agent backend: QwenAgents")
print("CUDA:", torch.cuda.is_available())
print("Seeds:", [run["seed"] for run in suite["by_seed"]])
print("Rounds per seed:", [len(run["results"]) for run in suite["by_seed"]])
print("F1:", [round(result["detection"]["f1"], 3) for run in suite["by_seed"] for result in run["results"]])

/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


[robustness] starting suite: seeds=2, rounds=2
[robustness] starting seed 20260821 with family plan: ['account_takeover', 'social_engineering']
[seed=20260821] starting 2-round run with family plan: ['account_takeover', 'social_engineering']
[seed=20260821] round 1/2 starting
[seed=20260821] round 1/2 complete | family=account_takeover | f1=0.0000 | novelty=1.0000
[seed=20260821] round 2/2 starting
[seed=20260821] round 2/2 complete | family=social_engineering | f1=0.1667 | novelty=0.9351
[seed=20260821] run complete. Total rounds: 2
[robustness] completed seed 20260821 with 2 rounds
[robustness] starting seed 20260822 with family plan: ['account_takeover', 'cross_channel_anomaly']
[seed=20260822] starting 2-round run with family plan: ['account_takeover', 'cross_channel_anomaly']
[seed=20260822] round 1/2 starting
[seed=20260822] round 1/2 complete | family=account_takeover | f1=0.3077 | novelty=0.9682
[seed=20260822] round 2/2 starting
[seed=20260822] round 2/2 complete | family=cros

In [9]:
import json
import os
import sys
import tempfile
from datetime import datetime, timezone

os.chdir(project_dir)
for module_name in list(sys.modules):
    if module_name == "mastercard_defence" or module_name.startswith("mastercard_defence."):
        del sys.modules[module_name]
sys.path.insert(0, str(project_dir / "src"))
from mastercard_defence.agents import QwenAgents
from mastercard_defence.llm import SharedLocalLLM
from mastercard_defence.loop import ClosedLoop, load_config

os.environ["RUN_MODE"] = "KAGGLE_GPU"
os.environ["MODEL_PATH"] = str(model_path)
config = load_config("config/default.yaml")
config["paths"]["memory_db"] = tempfile.mktemp(suffix=".sqlite")
config["generator_backend"] = "ctgan"
config["generator_epochs"] = 5
config["pipeline"]["synthetic_transactions"] = 400
config["pipeline"]["max_generated_attacks"] = 80

qwen_agents = QwenAgents(config, llm=SharedLocalLLM(config))
loop = ClosedLoop(config, agents=qwen_agents)
try:
    suite = loop.run_robustness_suite(seeds=3, rounds=5)
finally:
    loop.close()

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
result_path = project_dir / "artifacts" / f"qwen_ctgan_robustness_results_{run_stamp}.json"
result_path.parent.mkdir(parents=True, exist_ok=True)
artifact = {
    "run_timestamp_utc": run_stamp,
    "generator_backend": "conditional_ctgan",
    "agent_backend": "QwenAgents",
    "model_path": str(model_path),
    "seed_count": suite["seed_count"],
    "rounds": suite["rounds"],
    "summary": suite["summary"],
    "by_seed": [
        {"seed": run["seed"], "results": [
            {"round": item["round"], "attack_family": item["specification"].attack_family,
             "detection": item["detection"], "fidelity": item["fidelity"],
             "diversity": item["diversity"], "novelty": item["novelty"]}
            for item in run["results"]
        ]}
        for run in suite["by_seed"]
    ],
}
result_path.write_text(json.dumps(artifact, indent=2, default=str), encoding="utf-8")
print("FULL_QWEN_CTGAN_OK", suite["seed_count"], suite["rounds"])
print("CUDA:", torch.cuda.is_available())
print("Results saved:", result_path)
print("Summary:", suite["summary"])

/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


[robustness] starting suite: seeds=3, rounds=5
[robustness] starting seed 20260821 with family plan: ['account_takeover', 'social_engineering', 'trusted_device', 'beneficiary_manipulation', 'cross_channel_anomaly']
[seed=20260821] starting 5-round run with family plan: ['account_takeover', 'social_engineering', 'trusted_device', 'beneficiary_manipulation', 'cross_channel_anomaly']
[seed=20260821] round 1/5 starting
[seed=20260821] round 1/5 complete | family=account_takeover | f1=0.8531 | novelty=1.0000
[seed=20260821] round 2/5 starting
[seed=20260821] round 2/5 complete | family=social_engineering | f1=0.7669 | novelty=0.9439
[seed=20260821] round 3/5 starting
[seed=20260821] round 3/5 complete | family=trusted_device | f1=0.8345 | novelty=0.9375
[seed=20260821] round 4/5 starting
[seed=20260821] round 4/5 complete | family=beneficiary_manipulation | f1=0.8489 | novelty=0.9331
[seed=20260821] round 5/5 starting
[seed=20260821] round 5/5 complete | family=cross_channel_anomaly | f1=0.